# gate_diag_ZD3D

SAW with diagnostics: what the gate removes during training. Benchmark ZD3D, 3 run(s), about 120 min on a T4 GPU.

Files needed in `MyDrive/SAW_ablation/`: `zd3d_core.py`, `saw_ablation.py`.

Results are saved to `MyDrive/SAW_ablation/gate_diag_ZD3D/` after every run; re-running the notebook skips finished runs.

In [1]:
import os, sys, csv, time
import numpy as np, torch
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/SAW_ablation'
sys.path.insert(0, PROJECT_DIR)
import zd3d_core as core
import saw_ablation as sa

EXP, BENCH = 'gate_diag_ZD3D', 'ZD3D'
OUT = f'{PROJECT_DIR}/{EXP}'
os.makedirs(OUT, exist_ok=True)
device = 'cuda'
print(torch.cuda.get_device_name(0))

Mounted at /content/drive
Tesla T4


In [2]:
RUNS = [{'mode': 'saw', 'seed': 42, 'q': 0.95, 'diag': True}, {'mode': 'saw', 'seed': 43, 'q': 0.95, 'diag': True}, {'mode': 'saw', 'seed': 44, 'q': 0.95, 'diag': True}]
CSV_PATH = f'{OUT}/{EXP}.csv'
grid = core.Grid(n=sa.GRID_N)
PHI_E = None

In [3]:
def key(r):
    return (r['mode'], str(r['seed']), str(r.get('q', '')), str(r.get('w_eik', '')))

done = set()
if os.path.exists(CSV_PATH):
    with open(CSV_PATH) as f:
        done = {(r['mode'], r['seed'], r['q'], r['w_eik']) for r in csv.DictReader(f)}
todo = [r for r in RUNS if key(r) not in done]
print(f'{len(RUNS) - len(todo)} done, {len(todo)} to go')

for run in todo:
    run = dict(run)
    print(run['mode'], 'seed', run['seed'], flush=True)
    row, tag = sa.run_one(core, BENCH, run, device, OUT, grid, PHI_E)
    new = not os.path.exists(CSV_PATH)
    with open(CSV_PATH, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if new:
            w.writeheader()
        w.writerow(row)
    print(f"  rel L2 {row['avg_rel_L2']:.4f}%  MAPE {row['avg_vol_MAPE']:.4f}%  "
          f"final w {row['w_eik_final']}  {row['minutes']:.1f} min", flush=True)

0 done, 3 to go
saw seed 42


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


  rel L2 0.1581%  MAPE 0.4973%  final w 0.001511359354481101  34.8 min
saw seed 43
  rel L2 0.1925%  MAPE 1.0006%  final w 0.0028131434228271246  34.7 min
saw seed 44
  rel L2 0.5036%  MAPE 3.1859%  final w 0.002430463209748268  34.6 min


In [4]:
for r in csv.DictReader(open(CSV_PATH)):
    extra = f"  F_fill {float(r['avg_slot_filled']):.2f}%" if 'avg_slot_filled' in r else ''
    print(f"{r['mode']:7s} seed {r['seed']}  rel L2 {float(r['avg_rel_L2']):.4f}%  "
          f"MAPE {float(r['avg_vol_MAPE']):.4f}%{extra}  final w {r['w_eik_final']}")

saw     seed 42  rel L2 0.1581%  MAPE 0.4973%  F_fill 13.76%  final w 0.001511359354481101
saw     seed 43  rel L2 0.1925%  MAPE 1.0006%  F_fill 19.27%  final w 0.0028131434228271246
saw     seed 44  rel L2 0.5036%  MAPE 3.1859%  F_fill 26.10%  final w 0.002430463209748268
